# MyTravelHelper v3.0: Hướng dẫn tích hợp và kiểm thử VNPAY Sandbox

Notebook này minh họa chi tiết quy trình sinh mã chữ ký bảo mật (Secure Hash) bằng thuật toán **HMAC-SHA512** và xây dựng URL thanh toán đúng theo đặc tả kỹ thuật của cổng thanh toán VNPAY Sandbox (v2.1.0 API).

In [1]:
import hmac
import hashlib
import urllib.parse
from datetime import datetime

# Cấu hình thông tin kết nối VNPAY Sandbox (Mock credentials)
TMN_CODE = "DEMO_TMN"
HASH_SECRET = "DEMO_HASH_SECRET_1234567890"
VNPAY_URL = "https://sandbox.vnpayment.vn/paymentv2/vpcpay.html"
RETURN_URL = "http://localhost:8000/api/payment/return"

### 1. Sinh chữ ký bảo mật và URL thanh toán

Theo quy định kỹ thuật của VNPAY, các tham số gửi lên phải được:
1. Sắp xếp tăng dần theo tên key (alphabet).
2. Tạo chuỗi query dạng `key=value&key2=value2`.
3. Ký bằng HMAC-SHA512 với `HashSecret`.

In [2]:
def create_payment_url(order_id, amount, ip_address, description):
    create_date = datetime.now().strftime("%Y%m%d%H%M%S")
    
    # Các tham số bắt buộc của VNPAY
    vnp_params = {
        "vnp_Version": "2.1.0",
        "vnp_Command": "pay",
        "vnp_TmnCode": TMN_CODE,
        "vnp_Amount": str(int(amount * 100)), # Số tiền nhân 100
        "vnp_CreateDate": create_date,
        "vnp_CurrCode": "VND",
        "vnp_IpAddr": ip_address,
        "vnp_Locale": "vn",
        "vnp_OrderInfo": description,
        "vnp_OrderType": "other",
        "vnp_ReturnUrl": RETURN_URL,
        "vnp_TxnRef": order_id
    }
    
    # 1. Sắp xếp tham số theo bảng chữ cái A-Z
    sorted_params = sorted(vnp_params.items())
    
    # 2. Xây dựng chuỗi query data (mã hóa các ký tự đặc biệt)
    # VNPAY yêu cầu dấu cách được mã hóa thành dấu cộng '+'
    hash_data = []
    for key, val in sorted_params:
        hash_data.append(f"{key}={urllib.parse.quote_plus(str(val))}")
    query_string = "&".join(hash_data)
    
    # 3. Tính chữ ký HMAC-SHA512
    secure_hash = hmac.new(
        HASH_SECRET.encode('utf-8'),
        query_string.encode('utf-8'),
        hashlib.sha512
    ).hexdigest()
    
    # 4. Trả về liên kết thanh toán hoàn chỉnh
    final_url = f"{VNPAY_URL}?{query_string}&vnp_SecureHash={secure_hash}"
    return final_url, secure_hash

# Chạy sinh URL mẫu
test_url, test_hash = create_payment_url("DH202606071000", 500000.0, "127.0.0.1", "Thanh toan don hang sample")
print(f"Chữ ký Secure Hash: {test_hash}\n")
print(f"Đường dẫn thanh toán (redirection URL):\n{test_url}")

Chữ ký Secure Hash: f034689d3ae5992e3002180f007bf76230afd4251112b43ee51ab282111e7e3ec4acb0f29a4cca63215a6cc4c8f20d4b10e525f88bb35692621ce82c219e8e25

Đường dẫn thanh toán (redirection URL):
https://sandbox.vnpayment.vn/paymentv2/vpcpay.html?vnp_Amount=50000000&vnp_Command=pay&vnp_CreateDate=20260607100931&vnp_CurrCode=VND&vnp_IpAddr=127.0.0.1&vnp_Locale=vn&vnp_OrderInfo=Thanh+toan+don+hang+sample&vnp_OrderType=other&vnp_ReturnUrl=http%3A%2F%2Flocalhost%3A8000%2Fapi%2Fpayment%2Freturn&vnp_TmnCode=DEMO_TMN&vnp_TxnRef=DH202606071000&vnp_Version=2.1.0&vnp_SecureHash=f034689d3ae5992e3002180f007bf76230afd4251112b43ee51ab282111e7e3ec4acb0f29a4cca63215a6cc4c8f20d4b10e525f88bb35692621ce82c219e8e25


### 2. Xác thực phản hồi từ VNPAY (Callback / IPN)

Khi người dùng thanh toán xong, VNPAY chuyển hướng trình duyệt về `ReturnUrl` kèm các tham số thanh toán và chữ ký kiểm chứng `vnp_SecureHash`. Backend phải băm lại dữ liệu và so sánh với chữ ký để xác thực tính toàn vẹn.

In [3]:
def verify_vnpay_response(response_dict):
    if "vnp_SecureHash" not in response_dict:
        return False
        
    received_hash = response_dict["vnp_SecureHash"]
    
    # Tách chữ ký khỏi danh sách dữ liệu kiểm chứng
    vnp_data = {k: v for k, v in response_dict.items() if k.startswith("vnp_") and k != "vnp_SecureHash"}
    
    # Sắp xếp các tham số nhận được
    sorted_response = sorted(vnp_data.items())
    
    # Tạo chuỗi query
    hash_data = []
    for key, val in sorted_response:
        hash_data.append(f"{key}={urllib.parse.quote_plus(str(val))}")
    query_string = "&".join(hash_data)
    
    # Tính toán chữ ký so sánh
    calculated_hash = hmac.new(
        HASH_SECRET.encode('utf-8'),
        query_string.encode('utf-8'),
        hashlib.sha512
    ).hexdigest()
    
    is_valid = calculated_hash.lower() == received_hash.lower()
    print(f"Chữ ký tính toán: {calculated_hash}")
    print(f"Chữ ký nhận được: {received_hash}")
    print(f"Kết quả xác thực: {'HỢP LỆ' if is_valid else 'GIẢ MẠO / SAI CHỮ KÝ'}")
    return is_valid

# Giả lập dữ liệu thành công gửi về từ VNPAY
mock_callback = {
    "vnp_Amount": "50000000",
    "vnp_BankCode": "NCB",
    "vnp_BankTranNo": "VNP13579246",
    "vnp_CardType": "ATM",
    "vnp_OrderInfo": "Thanh toan don hang sample",
    "vnp_PayDate": "20260607100500",
    "vnp_ResponseCode": "00",
    "vnp_TmnCode": TMN_CODE,
    "vnp_TransactionNo": "12345678",
    "vnp_TxnRef": "DH202606071000"
}

# Tính toán SecureHash hợp lệ cho mock_callback để chạy thử
sorted_mock = sorted(mock_callback.items())
mock_query = "&".join([f"{k}={urllib.parse.quote_plus(v)}" for k, v in sorted_mock])
mock_hash = hmac.new(
    HASH_SECRET.encode('utf-8'),
    mock_query.encode('utf-8'),
    hashlib.sha512
).hexdigest()

mock_callback["vnp_SecureHash"] = mock_hash

# Xác thực
verify_vnpay_response(mock_callback)

Chữ ký tính toán: 2957f6fcd326f9a0d8139b2cbf393416b7ec680a0aeff38e296ef4f259640d18ca7fe0bb02bdd448f176b13d274a0dc96aa7d79a1a583155a0d2d23a09b12b04
Chữ ký nhận được: 2957f6fcd326f9a0d8139b2cbf393416b7ec680a0aeff38e296ef4f259640d18ca7fe0bb02bdd448f176b13d274a0dc96aa7d79a1a583155a0d2d23a09b12b04
Kết quả xác thực: HỢP LỆ


True